# 🚀 Space Station 14 — Autonomous OpenAI Crew Server & Live Web CCTV (NSS Saltern)

Этот ноутбук полностью разворачивает в **Google Colab**:
1. **Настоящий сервер Space Station 14 (`Content.Server`)** с интегрированным C#-модом `Content.Server/AIStation` (`AiAgentComponent` + `AiStationBridgeSystem`) и реальной картой станции **NSS Saltern** (`saltern.yml`: 6905 тайлов, 437 шлюзов, 53 камеры СБ, 56 маяков отсеков, 1004 оригинальных `.rsi` спрайта из открытого исходника `space-wizards/space-station-14`).
2. **Мульти-агентный OpenAI-совместимый ИИ-оркестратор**: каждый член экипажа (Капитан, ГП, ГСБ, Офицер СБ, Детектив, Старший Инженер, Атмосферный техник, Главврач, Химик, Научный руководитель, Квартирмейстер, Бармен, Уборщик, Клоун-агент Синдиката) управляется через любой **OpenAI-compatible API** (`/v1/chat/completions`) и искренне верит, что он живой человек на станции.
3. **Веб-центр наблюдения и управления с камерами СБ**: позволяет прямо в браузере (или прямо внутри ячейки ноутбука!) вводить `API Key` и `Base URL Endpoint`, переключаться между 53 камерами станции и режимом сетки 4х камер (CCTV Matrix), следить за мыслями, памятью и чатом ИИ, внушать мысли персонажам или **брать любого персонажа под прямое управление (WASD + Мышь)**.

In [ ]:
#@title 1️⃣ Клонирование репозитория howlol/ss144 и установка зависимостей
import os, subprocess, sys

REPO_URL = "https://github.com/howlol/ss144.git"
BRANCH = "arena/01a101dd-ss144"
WORK_DIR = "/content/ss144"

if not os.path.exists(WORK_DIR):
    res = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR])
    if res.returncode != 0:
        subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=False)

os.chdir(WORK_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Установим cloudflared для публичного HTTPS-доступа к веб-панели и камерам из любого браузера
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run([
        "bash", "-c",
        "curl -sSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
    ], check=True)

print("✅ Репозиторий и зависимости успешно установлены в", os.getcwd())

In [ ]:
#@title 2️⃣ Настройка OpenAI API (можно также менять в любой момент прямо в Веб-интерфейсе) и сборка C# сервера SS14
import os, subprocess

OPENAI_BASE_URL = "https://api.openai.com/v1" #@param {type:"string"}
OPENAI_API_KEY = "" #@param {type:"string"}
OPENAI_MODEL = "gpt-4o-mini" #@param {type:"string"}
BUILD_NATIVE_CSHARP_CONTENT_SERVER = False #@param {type:"boolean"}

os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL.strip()
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
os.environ["OPENAI_MODEL"] = OPENAI_MODEL.strip()

if BUILD_NATIVE_CSHARP_CONTENT_SERVER:
    print("🔨 Сборка нативного C# Content.Server (space-wizards/space-station-14 + AIStation mod)...")
    subprocess.run(["bash", "scripts/build_ss14_server.sh"], check=True)
    print("🚀 Запуск скомпилированного C# Content.Server в фоне (Bridge :12120 / Game UDP :1212)...")
    subprocess.Popen(
        ["/tmp/ss14-src/bin/Content.Server/Content.Server"],
        cwd="/tmp/ss14-src",
        stdout=open("/tmp/ss14_csharp.log", "w"),
        stderr=subprocess.STDOUT,
    )
else:
    print("⚡ Выбран быстрый старт: реальная карта NSS Saltern (saltern.yml) и 1004 оригинальных .rsi текстур уже загружены из репозитория!")

In [ ]:
#@title 3️⃣ Запуск сервера SS14 AI Station, камер наблюдения и открытие панели управления
import re, subprocess, time, urllib.request

# Остановим предыдущий процесс на порту 8000, если он был запущен
subprocess.run(["pkill", "-f", "uvicorn server.app:app"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(1)

server_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("/tmp/ss14_web.log", "w"),
    stderr=subprocess.STDOUT,
)

# Ждём готовности сервера
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/config", timeout=1) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(0.5)

# Запускаем публичный HTTPS-туннель Cloudflare
cf_log = open("/tmp/cloudflared.log", "w+")
cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    cf_log.seek(0)
    content = cf_log.read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.5)

print("=" * 78)
print("🛰️ СЕРВЕР SPACE STATION 14 (NSS SALTERN) И КАМЕРЫ НАБЛЮДЕНИЯ ЗАПУЩЕНЫ!")
if public_url:
    print(f"🌐 ПУБЛИЧНАЯ ССЫЛКА ДЛЯ БРАУЗЕРА: {public_url}")
print("💡 Вы можете ввести или поменять OpenAI API Key и Endpoint прямо в браузере (кнопка '🔑 OpenAI API & Endpoint')")
print("=" * 78)

# Отобразим интерактивный экран с камерами прямо внутри Google Colab!
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(8000, height='820')
except ImportError:
    print("Откройте http://localhost:8000 в браузере.")

In [ ]:
#@title 4️⃣ Быстрая сводка по станции и чату экипажа прямо в блокноте
import json, urllib.request

with urllib.request.urlopen("http://127.0.0.1:8000/api/state") as resp:
    st = json.loads(resp.read().decode())

print(f"⏱️ Время раунда: {st['roundClock']} | Код тревоги: {st['alertLevel'].upper()} | Питание: {st['stationPowerKw']} kW")
print("\n👥 ЭКИПАЖ ИИ НА СТАНЦИИ NSS SALTERN:")
for a in st["agents"]:
    print(f"  • {a['name']} ({a['jobTitleRu']}) — 📍 {a['currentRoom']} | ⚡ {a['currentTask']} | 💭 '{a['lastThought']}'")

print("\n📻 ПОСЛЕДНИЕ СООБЩЕНИЯ В РАЦИИ И ЛОКАЛЬНОМ ЧАТЕ:")
for c in st["chat"][-10:]:
    print(f"  [{c['time']}] [{c['channel']}] {c['speakerName']}: {c['message']}")